# HCLA RQ3 — Revised Explanation Quality Experiment

**Conditions:** Numerical / Feature Attribution / Template / HCLA

**Primary metrics**
1. **Clarity** — How clearly the explanation is presented
2. **Evidence Integration** — How well evidence is integrated into the prediction explanation rather than merely listed
3. **Calibrated Usefulness** — How well the explanation helps interpret the model decision without overstatement

RQ2 metrics (**ESR/UCR/CR/EC**) are not evaluated again. All four methods use the same cases and evidence.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# HCLA RQ3 - Explanation Quality Comparison (Revised)
# Conditions: Numerical / Feature Attribution / Template / HCLA
# Metrics: Clarity / Evidence Integration / Calibrated Usefulness
#
# Design principles
# 1) Same cases, same prediction, same SHAP evidence across all 4 conditions
# 2) Condition is the only manipulated factor
# 3) RQ2 groundedness metrics (ESR/UCR/CR/EC) are NOT duplicated here
# 4) Judge sees explanation + common evidence package, but NOT condition name
# 5) Explicit 1-5 anchors reduce ceiling effects and identical-score collapse
# ============================================================

import os, json, re, time, random, pickle, getpass, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [ ]:
# -----------------------------
# 0. Reproducibility / settings

In [ ]:
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# PILOT: 10 cases -> 40 explanations -> 40 judge calls
# FULL : 50 cases -> 200 explanations -> 200 judge calls
RQ3_PILOT = False
RQ3_PILOT_N = 10
RQ3_FULL_N = 50

# Keep class balance when evidence package contains both classes
BALANCED_SAMPLING = True

# API
QWEN_MODEL = "qwen/qwen3-235b-a22b-2507"
LLAMA_JUDGE_MODEL = "meta-llama/llama-3.3-70b-instruct"
GEN_TEMPERATURE = 0.2
JUDGE_TEMPERATURE = 0.0
QWEN_WORKERS = 5
JUDGE_WORKERS = 5
CHECKPOINT_EVERY = 10

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

In [ ]:
# -----------------------------
# 1. Paths

In [ ]:
# -----------------------------
DEFAULT_BASE = "/content/drive/MyDrive/Colab Notebooks/Results/HCLA_MultiModel_Qwen_Llama_OpenRouter_SHAP"
BASE_DIR = os.environ.get("HCLA_BASE_DIR", DEFAULT_BASE)

# RQ2 creates this file after SHAP evidence extraction.
# You can override via environment variable:
#   os.environ["RQ3_EVIDENCE_PKL"] = "/your/path/evidence_packages.pkl"
DEFAULT_EVIDENCE_CANDIDATES = [
    os.environ.get("RQ3_EVIDENCE_PKL", ""),
    os.path.join(BASE_DIR, "_checkpoints", "evidence_packages.pkl"),
    #os.path.join(BASE_DIR, "RQ2", "_checkpoints", "evidence_packages.pkl"),
    #os.path.join(BASE_DIR, "rq2", "_checkpoints", "evidence_packages.pkl"),
]

In [ ]:
DEFAULT_EVIDENCE_CANDIDATES

In [ ]:
RQ3_OUTPUT_DIR = os.path.join(BASE_DIR, "RQ3_Revised")
CHECKPOINT_DIR = os.path.join(RQ3_OUTPUT_DIR, "_checkpoints")

In [ ]:
CHECKPOINT_DIR

In [ ]:
BASE_DIR

In [ ]:
from pathlib import Path
import os

DEFAULT_BASE = "/content/drive/MyDrive/Colab Notebooks/Results/HCLA_MultiModel_Qwen_Llama_SHAP"
BASE_DIR = os.environ.get("HCLA_BASE_DIR", DEFAULT_BASE)

DEFAULT_EVIDENCE_CANDIDATES = [
    os.environ.get("RQ3_EVIDENCE_PKL", ""),

    # Actual directory name: _checkpoint
    os.path.join(BASE_DIR, "_checkpoints", "evidence_packages.pkl"),
]

RQ3_OUTPUT_DIR = os.path.join(BASE_DIR, "RQ3_Revised")
CHECKPOINT_DIR = os.path.join(RQ3_OUTPUT_DIR, "_checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RQ3_OUTPUT_DIR, exist_ok=True)


def find_existing(candidates):
    for p in candidates:
        if p and os.path.exists(p):
            return p
    return None


EVIDENCE_PKL = find_existing(DEFAULT_EVIDENCE_CANDIDATES)

# Search across BASE_DIR even if the exact location differs.
if EVIDENCE_PKL is None:
    matches = list(Path(BASE_DIR+"/_checkpoints").rglob("evidence_packages.pkl"))
    if matches:
        EVIDENCE_PKL = str(matches[0])


if EVIDENCE_PKL is None:
    raise FileNotFoundError(
        f"Could not find evidence_packages.pkl.\n"
        f"Search base directory: {BASE_DIR}"
    )


print("✅ Evidence:", EVIDENCE_PKL)
print("✅ Output  :", RQ3_OUTPUT_DIR)

In [ ]:
BASE_DIR

In [ ]:
with open(EVIDENCE_PKL, "rb") as f:
    evidence_packages = pickle.load(f)

if not isinstance(evidence_packages, list) or len(evidence_packages) == 0:
    raise ValueError("Unexpected evidence_packages.pkl format.")

print("Evidence packages loaded:", len(evidence_packages))
print("Example keys:", evidence_packages[0].keys())

REQUIRED_KEYS = {
    "case_id",
    "true_label",
    "prediction",
    "top_evidence"
}

missing = REQUIRED_KEYS - set(evidence_packages[0].keys())

if missing:
    raise ValueError(f"Evidence package missing keys: {missing}")

print("✅ Evidence package structure OK")

In [ ]:
# -----------------------------
# 3. Select SAME cases for all 4 methods

In [ ]:
# ============================================================
# RQ3 Balanced Sampling
# Handle true_label strings: Normal / Anomaly.
# ============================================================

import numpy as np


def normalize_label(v):
    """Normalize Normal/Anomaly and 0/1 labels to 0/1."""
    if isinstance(v, str):
        s = v.strip().lower()

        if s in ("normal", "0"):
            return 0

        if s in ("anomaly", "anomalous", "1"):
            return 1

    try:
        return int(v)
    except (ValueError, TypeError):
        return -1


def balanced_select(packages, n, seed=42):
    rng = np.random.default_rng(seed)

    if not BALANCED_SAMPLING:
        idx = rng.choice(
            len(packages),
            size=min(n, len(packages)),
            replace=False
        )
        return [packages[i] for i in idx]

    normal = [
        e for e in packages
        if normalize_label(e.get("true_label")) == 0
    ]

    anomaly = [
        e for e in packages
        if normalize_label(e.get("true_label")) == 1
    ]

    print(f"Available → Normal: {len(normal)}, Anomaly: {len(anomaly)}")

    n0 = n // 2
    n1 = n - n0

    if len(normal) < n0 or len(anomaly) < n1:
        raise ValueError(
            f"Balanced sample impossible: "
            f"normal={len(normal)}, anomaly={len(anomaly)}, requested={n}"
        )

    idx0 = rng.choice(len(normal), size=n0, replace=False)
    idx1 = rng.choice(len(anomaly), size=n1, replace=False)

    out = (
        [normal[i] for i in idx0]
        + [anomaly[i] for i in idx1]
    )

    rng.shuffle(out)

    return out


# ============================================================
# Select RQ3 cases
# ============================================================

N_CASES = RQ3_PILOT_N if RQ3_PILOT else RQ3_FULL_N

rq3_cases = balanced_select(
    evidence_packages,
    N_CASES,
    SEED
)

print()
print(f"RQ3 mode: {'PILOT' if RQ3_PILOT else 'FULL'}")
print("Cases:", len(rq3_cases))
print(
    "Normal:",
    sum(normalize_label(e["true_label"]) == 0 for e in rq3_cases)
)
print(
    "Anomaly:",
    sum(normalize_label(e["true_label"]) == 1 for e in rq3_cases)
)
print("Expected explanation conditions:", len(rq3_cases) * 4)

In [ ]:
# -----------------------------
# 4. Helpers

In [ ]:
# -----------------------------
def atomic_csv_save(df, path):
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(tmp, path)

def load_csv_or_empty(path, cols):
    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            for c in cols:
                if c not in df.columns:
                    df[c] = np.nan
            return df[cols]
        except Exception:
            pass
    return pd.DataFrame(columns=cols)

def safe_float(x, ndigits=4):
    try:
        return round(float(x), ndigits)
    except Exception:
        return x

def evidence_rows(e):
    """Normalize evidence rows from RQ2 evidence package."""
    rows = e.get("evidence", [])
    norm = []
    for r in rows:
        norm.append({
            "feature": r.get("feature", r.get("name", "")),
            "value": safe_float(r.get("value", np.nan)),
            "shap": safe_float(r.get("shap", r.get("shap_value", np.nan))),
            "effect": r.get("effect", ""),
        })
    return norm

def probability_text(e):
    p = e.get("anomaly_probability", None)
    if p is None:
        return "not available"
    try:
        return f"{float(p):.4f}"
    except Exception:
        return str(p)

In [ ]:
# -----------------------------
# 5. OpenRouter

In [ ]:
# -----------------------------
try:
    from openai import OpenAI
except ImportError:
    raise ImportError("The openai package is required: pip install openai")

def load_openrouter_key():
    """Read OpenRouter credentials without embedding them in this notebook."""
    key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    if not key:
        try:
            from google.colab import userdata
            key = (userdata.get("OPENROUTER_API_KEY") or "").strip()
        except (ImportError, KeyError, AttributeError):
            pass
    if not key:
        raise RuntimeError(
            "Set OPENROUTER_API_KEY as an environment variable or a Colab Secret before running."
        )
    return key

OPENROUTER_API_KEY = load_openrouter_key()

def make_client():
    return OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
        timeout=120.0,
        max_retries=0,
    )

def call_model(model, prompt, temperature=0.0, max_retries=5):
    client = make_client()
    for attempt in range(max_retries):
        try:
            r = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            text = r.choices[0].message.content
            if not text:
                raise RuntimeError("empty response")
            return text.strip()
        except Exception as ex:
            if attempt >= max_retries - 1:
                raise
            wait = min(60, 5 * (2 ** attempt))
            print(f"[retry {attempt+1}/{max_retries}] {type(ex).__name__}: {ex} -> {wait}s")
            time.sleep(wait)

def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.I).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            raise ValueError(text[:1000])
        return json.loads(m.group(0))

In [ ]:
# -----------------------------
# 6. FOUR explanation conditions

In [ ]:
# -----------------------------
# IMPORTANT:
# All four conditions use the SAME prediction/case.
# The difference is explanation representation only.

def make_numerical(e):
    """Score/prediction only. No SHAP attribution."""
    return (
        f"Model prediction: {e['prediction']}. "
        f"Anomaly probability: {probability_text(e)}."
    )

def make_feature_attribution(e):
    """Raw SHAP attribution list. No prose interpretation."""
    rows = evidence_rows(e)
    items = []
    for r in rows:
        items.append(
            f"{r['feature']}={r['value']} (SHAP={r['shap']}, effect={r['effect']})"
        )
    return (
        f"Model prediction: {e['prediction']}; anomaly probability: {probability_text(e)}. "
        "Top feature attributions: " + "; ".join(items) + "."
    )

def make_template(e):
    """Fixed, deterministic template using identical SHAP evidence."""
    rows = evidence_rows(e)
    if not rows:
        return (
            f"The model predicted {e['prediction']} with anomaly probability "
            f"{probability_text(e)}. No feature attribution was available."
        )

    positive = [r for r in rows if str(r["effect"]).lower() in {"increase", "positive", "anomaly", "higher"} or (isinstance(r["shap"], (int,float)) and r["shap"] > 0)]
    negative = [r for r in rows if isinstance(r["shap"], (int,float)) and r["shap"] < 0]

    top = rows[:3]
    top_txt = ", ".join(
        f"{r['feature']} (value={r['value']}, SHAP={r['shap']})" for r in top
    )

    return (
        f"The model predicted {e['prediction']} with anomaly probability "
        f"{probability_text(e)}. The strongest attributed features were {top_txt}. "
        "These feature attributions indicate which inputs most influenced the model output; "
        "they do not by themselves establish a causal or criminal interpretation."
    )

HCLA_PROMPT = """You are the HCLA Expert-Style Justification component.

Generate a concise explanation of the model prediction using ONLY the supplied evidence.
Your explanation must:
1. State the model prediction and, if available, anomaly probability.
2. Integrate the most relevant SHAP evidence into a coherent explanation rather than merely listing features.
3. Distinguish model evidence from real-world causal or criminal conclusions.
4. Avoid unsupported external facts, address history, intent, identity, or causal claims.
5. State uncertainty/limitation when the evidence cannot justify a stronger conclusion.
6. Use 2-4 sentences.

Evidence package:
{evidence}

Return ONLY the explanation text.
"""

def make_hcla(e):
    return call_model(
        QWEN_MODEL,
        HCLA_PROMPT.format(evidence=json.dumps(e, ensure_ascii=False, indent=2)),
        temperature=GEN_TEMPERATURE,
        max_retries=5,
    )

METHODS = ["Numerical", "Feature Attribution", "Template", "HCLA"]

In [ ]:
# -----------------------------
# 7. Generate explanations

In [ ]:
# -----------------------------
GEN_CKPT = os.path.join(CHECKPOINT_DIR, "rq3_explanations.csv")
GEN_COLS = [
    "case_id", "true_label", "prediction", "method",
    "explanation", "evidence_json"
]

gen_df = load_csv_or_empty(GEN_CKPT, GEN_COLS)
rows = gen_df.to_dict("records") if len(gen_df) else []
done = {(str(r["case_id"]), str(r["method"])) for r in rows}

# Deterministic 3 baselines first
for e in rq3_cases:
    for method, fn in [
        ("Numerical", make_numerical),
        ("Feature Attribution", make_feature_attribution),
        ("Template", make_template),
    ]:
        key = (str(e["case_id"]), method)
        if key in done:
            continue
        rows.append({
            "case_id": e["case_id"],
            "true_label": e["true_label"],
            "prediction": e["prediction"],
            "method": method,
            "explanation": fn(e),
            "evidence_json": json.dumps(e, ensure_ascii=False),
        })
        done.add(key)

atomic_csv_save(pd.DataFrame(rows, columns=GEN_COLS), GEN_CKPT)

# HCLA in parallel
hcla_jobs = [e for e in rq3_cases if (str(e["case_id"]), "HCLA") not in done]

def hcla_job(e):
    return e, make_hcla(e)

if hcla_jobs:
    with ThreadPoolExecutor(max_workers=QWEN_WORKERS) as ex:
        futures = [ex.submit(hcla_job, e) for e in hcla_jobs]
        completed = 0
        for fut in as_completed(futures):
            e, explanation = fut.result()
            rows.append({
                "case_id": e["case_id"],
                "true_label": e["true_label"],
                "prediction": e["prediction"],
                "method": "HCLA",
                "explanation": explanation,
                "evidence_json": json.dumps(e, ensure_ascii=False),
            })
            done.add((str(e["case_id"]), "HCLA"))
            completed += 1
            if completed % CHECKPOINT_EVERY == 0:
                atomic_csv_save(pd.DataFrame(rows, columns=GEN_COLS), GEN_CKPT)
            print(f"✅ HCLA generated {completed}/{len(hcla_jobs)}")

explanations_df = pd.DataFrame(rows, columns=GEN_COLS)
explanations_df = explanations_df[
    explanations_df["case_id"].astype(str).isin({str(e["case_id"]) for e in rq3_cases})
].copy()
explanations_df["method"] = pd.Categorical(
    explanations_df["method"], categories=METHODS, ordered=True
)
explanations_df = explanations_df.sort_values(["case_id", "method"]).reset_index(drop=True)
atomic_csv_save(explanations_df, GEN_CKPT)

print("Explanations:", len(explanations_df))
print(explanations_df.groupby("method", observed=False).size())

In [ ]:
# -----------------------------
# 8. Revised RQ3 judge

In [ ]:
# ============================================================
# RQ3 - Blind LLM Judge
# Robust version for OpenRouter 402 / JSON parsing / checkpoint
# ============================================================

import os
import json
import re
import time
import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed

# ------------------------------------------------------------
# Reliability settings
# ------------------------------------------------------------

# Avoid OpenRouter in-flight budget errors.
# Start with 1; increase to 2 after confirming stable operation.
JUDGE_WORKERS = 1

# Require brief judge outputs; do not pass max_tokens to call_model() in this block.

# Maximum final retries per case
JUDGE_OUTER_RETRIES = 4

# Allow extra time because OpenRouter may respond with Retry-After: 120.
JUDGE_402_WAIT = 125

# Retry JSON parsing errors.
JUDGE_PARSE_WAIT = 5


# ============================================================
# Judge Prompt
# ============================================================

RQ3_JUDGE = """You are a strict evaluator of AI-generated explanations for an anomaly detector.

Evaluate the explanation ONLY using the supplied common evidence package.
The explanation method name is intentionally hidden.

Score THREE DISTINCT dimensions from 1 to 5.
Do NOT give the same score automatically across dimensions.
Use the anchors literally.

A. CLARITY
1 = confusing, fragmented, or largely uninterpretable
2 = understandable only with substantial effort; major wording/structure problems
3 = generally understandable but somewhat technical, vague, or poorly organized
4 = clear and coherent with only minor issues
5 = exceptionally clear, concise, and easy to understand

B. EVIDENCE_INTEGRATION
1 = gives little/no usable connection between the prediction and supplied evidence
2 = mentions a score or evidence superficially without explaining its relevance
3 = identifies relevant evidence but mostly lists/describes it
4 = clearly connects multiple supplied evidence items to the model output
5 = integrates the strongest supplied evidence into a coherent, specific explanation of the model output

C. CALIBRATED_USEFULNESS
1 = not useful for interpretation, or makes misleading/overconfident conclusions
2 = limited practical interpretive value; important caveats or context are missing
3 = moderately useful; supports basic interpretation but has notable limitations
4 = useful for reviewing the model output and appropriately cautious
5 = highly useful for interpretation, prioritizes what matters, and clearly separates model evidence from unsupported real-world conclusions

IMPORTANT:
- Judge explanation quality, NOT whether the model prediction is correct.
- Do not reward verbosity.
- A feature list may score well on factual specificity but should NOT receive high Evidence Integration unless relationships are actually synthesized.
- A numerical score alone should NOT receive high Evidence Integration.
- Do not infer external AML/criminal facts not present in evidence.
- Use the full 1-5 scale when warranted.
- Give each dimension independently.

Return EXACTLY SIX lines in this format.
Do NOT return JSON, markdown, bullets, or additional text.
The first three values MUST be a single integer from 1 to 5.
Do not use the "|" character inside reasons.

CLARITY|4
EVIDENCE_INTEGRATION|3
CALIBRATED_USEFULNESS|4
CLARITY_REASON|brief reason
EVIDENCE_INTEGRATION_REASON|brief reason
CALIBRATED_USEFULNESS_REASON|brief reason
"""


# ============================================================
# Check whether the error is HTTP 402.
# ============================================================

def is_402_error(e):
    """
    Check for HTTP 402 in OpenRouter / OpenAI APIStatusError
    """
    status = getattr(e, "status_code", None)

    if status == 402:
        return True

    msg = str(e).lower()

    return (
        "error code: 402" in msg
        or "in_flight_budget_exhausted" in msg
        or "openrouter_in_flight_budget" in msg
    )


# ============================================================
# Evaluate one case with the judge.
# ============================================================

def parse_rq3_judge_output(raw):
    """
    Parse the six-line delimiter format.
    Invalid/malformed outputs raise ValueError so the case is retried.
    """
    if raw is None:
        raise ValueError("Judge returned None")

    raw = str(raw).strip()

    # Remove accidental markdown fences without accepting arbitrary prose.
    raw = re.sub(r"^```(?:text)?\s*", "", raw, flags=re.I)
    raw = re.sub(r"\s*```$", "", raw)

    values = {}

    allowed = {
        "CLARITY",
        "EVIDENCE_INTEGRATION",
        "CALIBRATED_USEFULNESS",
        "CLARITY_REASON",
        "EVIDENCE_INTEGRATION_REASON",
        "CALIBRATED_USEFULNESS_REASON",
    }

    for line in raw.splitlines():
        line = line.strip()
        if not line or "|" not in line:
            continue

        key, value = line.split("|", 1)
        key = key.strip().upper()
        value = value.strip()

        if key in allowed and key not in values:
            values[key] = value

    required_scores = [
        "CLARITY",
        "EVIDENCE_INTEGRATION",
        "CALIBRATED_USEFULNESS",
    ]

    missing = [k for k in required_scores if k not in values]
    if missing:
        raise ValueError(f"Missing judge score fields: {missing}; raw={raw[:500]!r}")

    scores = {}
    for key in required_scores:
        # STRICT: only one integer 1..5 is accepted.
        # Never convert a reason sentence into a score.
        m = re.fullmatch(r"[1-5]", values[key])
        if not m:
            raise ValueError(
                f"Invalid {key} score: {values[key]!r}; raw={raw[:500]!r}"
            )
        scores[key] = int(values[key])

    return {
        "clarity": scores["CLARITY"],
        "evidence_integration": scores["EVIDENCE_INTEGRATION"],
        "calibrated_usefulness": scores["CALIBRATED_USEFULNESS"],
        "clarity_reason": values.get("CLARITY_REASON", ""),
        "evidence_integration_reason": values.get("EVIDENCE_INTEGRATION_REASON", ""),
        "calibrated_usefulness_reason": values.get("CALIBRATED_USEFULNESS_REASON", ""),
    }


def judge_one(row):

    evidence = json.loads(row["evidence_json"])

    # method/condition is intentionally hidden from the judge.
    prompt = f"""{RQ3_JUDGE}

COMMON EVIDENCE PACKAGE:
{json.dumps(evidence, ensure_ascii=False, indent=2)}

EXPLANATION:
{row['explanation']}
"""

    last_error = None

    for attempt in range(1, JUDGE_OUTER_RETRIES + 1):
        try:
            # IMPORTANT:
            # The project's call_model() does NOT accept max_tokens here.
            # Therefore do not pass max_tokens in this RQ3 block.
            raw = call_model(
                LLAMA_JUDGE_MODEL,
                prompt,
                temperature=JUDGE_TEMPERATURE,
                max_retries=2,
            )

            obj = parse_rq3_judge_output(raw)

            return {
                "case_id": row["case_id"],
                "true_label": row["true_label"],
                "prediction": row["prediction"],
                "method": str(row["method"]),
                "clarity": obj["clarity"],
                "evidence_integration": obj["evidence_integration"],
                "calibrated_usefulness": obj["calibrated_usefulness"],
                "clarity_reason": obj.get("clarity_reason", ""),
                "evidence_integration_reason": obj.get("evidence_integration_reason", ""),
                "calibrated_usefulness_reason": obj.get("calibrated_usefulness_reason", ""),
            }

        except Exception as e:
            last_error = e

            if is_402_error(e):
                if attempt < JUDGE_OUTER_RETRIES:
                    print(
                        f"⚠️ 402 in-flight budget "
                        f"[attempt {attempt}/{JUDGE_OUTER_RETRIES}]"
                    )
                    print(f"   → {JUDGE_402_WAIT}s before retrying")
                    time.sleep(JUDGE_402_WAIT)
                    continue

            if attempt < JUDGE_OUTER_RETRIES:
                print(
                    f"⚠️ Judge retry "
                    f"[{attempt}/{JUDGE_OUTER_RETRIES}] "
                    f"{type(e).__name__}: {e}"
                )
                time.sleep(JUDGE_PARSE_WAIT)
                continue

            break

    raise RuntimeError(
        f"Judge failed after {JUDGE_OUTER_RETRIES} attempts | "
        f"case_id={row['case_id']} | method={row['method']} | "
        f"last_error={last_error}"
    )


# ============================================================
# Checkpoint settings
# ============================================================

JUDGE_CKPT = os.path.join(
    CHECKPOINT_DIR,
    "rq3_judgments.csv"
)

JUDGE_COLS = [
    "case_id",
    "true_label",
    "prediction",
    "method",
    "clarity",
    "evidence_integration",
    "calibrated_usefulness",
    "clarity_reason",
    "evidence_integration_reason",
    "calibrated_usefulness_reason",
]


# ============================================================
# Load existing checkpoints.
# ============================================================

judge_df0 = load_csv_or_empty(
    JUDGE_CKPT,
    JUDGE_COLS
)

judge_rows = (
    judge_df0.to_dict("records")
    if len(judge_df0)
    else []
)

judge_done = {
    (
        str(r["case_id"]),
        str(r["method"])
    )
    for r in judge_rows
}

print(
    f"✅ Existing judgments: {len(judge_rows)}"
)


# ============================================================
# Select only explanations that have not been evaluated.
# ============================================================

todo = []

for _, row in explanations_df.iterrows():

    key = (
        str(row["case_id"]),
        str(row["method"])
    )

    if key not in judge_done:
        todo.append(row.to_dict())


print(
    f"✅ Remaining judgments: {len(todo)}"
)

print(
    f"✅ Judge workers: {JUDGE_WORKERS}"
)

print(
    "✅ Judge output format: strict 6-line delimiter (no model-generated JSON)"
)


# ============================================================
# Run the judge.
# ============================================================

failed_rows = []

if todo:

    with ThreadPoolExecutor(
        max_workers=JUDGE_WORKERS
    ) as ex:

        future_to_row = {
            ex.submit(judge_one, row): row
            for row in todo
        }

        completed = 0

        for fut in as_completed(future_to_row):

            source_row = future_to_row[fut]

            try:

                out = fut.result()

                judge_rows.append(out)

                judge_done.add(
                    (
                        str(out["case_id"]),
                        str(out["method"])
                    )
                )

                completed += 1

                # -----------------------------------------
                # Save checkpoints as frequently as feasible.
                # -----------------------------------------

                if (
                    completed % CHECKPOINT_EVERY == 0
                    or completed == len(todo)
                ):

                    atomic_csv_save(
                        pd.DataFrame(
                            judge_rows,
                            columns=JUDGE_COLS
                        ),
                        JUDGE_CKPT
                    )

                print(
                    f"✅ Judge "
                    f"{completed}/{len(todo)}"
                )

            except Exception as e:

                # -----------------------------------------
                # Do not terminate the entire experiment because of one failed case.
                # -----------------------------------------

                print(
                    "\n❌ Judge final failure"
                )

                print(
                    "case_id:",
                    source_row.get("case_id")
                )

                print(
                    "method:",
                    source_row.get("method")
                )

                print(
                    "error:",
                    repr(e)
                )

                failed_rows.append({
                    "case_id":
                        source_row.get("case_id"),

                    "method":
                        str(source_row.get("method")),

                    "error":
                        str(e),
                })

                # Save successful results first.
                atomic_csv_save(
                    pd.DataFrame(
                        judge_rows,
                        columns=JUDGE_COLS
                    ),
                    JUDGE_CKPT
                )


# ============================================================
# Final DataFrame
# ============================================================

judgments_df = pd.DataFrame(
    judge_rows,
    columns=JUDGE_COLS
)


# Retain only the current rq3_cases.
valid_case_ids = {
    str(e["case_id"])
    for e in rq3_cases
}

judgments_df = judgments_df[
    judgments_df["case_id"]
    .astype(str)
    .isin(valid_case_ids)
].copy()


# Use a fixed method order.
judgments_df["method"] = pd.Categorical(
    judgments_df["method"],
    categories=METHODS,
    ordered=True
)


judgments_df = (
    judgments_df
    .sort_values(
        ["case_id", "method"]
    )
    .reset_index(drop=True)
)


# Save final results.
atomic_csv_save(
    judgments_df,
    JUDGE_CKPT
)


# ============================================================
# Integrity checks
# ============================================================

expected_n = len(explanations_df)
actual_n = len(judgments_df)

print()
print("=" * 60)
print("RQ3 JUDGE SUMMARY")
print("=" * 60)

print(
    f"Expected judgments : {expected_n}"
)

print(
    f"Completed judgments: {actual_n}"
)

print(
    f"Failed judgments   : {len(failed_rows)}"
)


# Counts by method
if len(judgments_df):

    print()
    print("Judgments by method:")

    print(
        judgments_df["method"]
        .value_counts(sort=False)
    )


# Save failed cases.
if failed_rows:

    FAILED_PATH = os.path.join(
        CHECKPOINT_DIR,
        "rq3_judge_failures.csv"
    )

    pd.DataFrame(
        failed_rows
    ).to_csv(
        FAILED_PATH,
        index=False
    )

    print()
    print(
        f"⚠️ Failed items saved: {FAILED_PATH}"
    )

    print(
        "⚠️ Some items failed. Before computing manuscript statistics, "
        "rerun this cell."
    )

else:

    if actual_n == expected_n:

        print()
        print(
            "✅ ALL RQ3 JUDGMENTS COMPLETE"
        )

    else:

        print()
        print(
            "⚠️ Completed count differs from expected count."
        )


print(
    f"✅ Checkpoint: {JUDGE_CKPT}"
)

In [ ]:
# -----------------------------
# 9. Summary + discrimination diagnostics

In [ ]:
# -----------------------------
METRICS = ["clarity", "evidence_integration", "calibrated_usefulness"]

summary = judgments_df.groupby("method", observed=False)[METRICS].agg(["mean", "std", "median"])
summary.columns = ["_".join(x) for x in summary.columns]
summary = summary.reset_index()

# Composite score is secondary, not a replacement for three dimensions.
judgments_df["composite"] = judgments_df[METRICS].mean(axis=1)

composite_summary = judgments_df.groupby("method", observed=False)["composite"].agg(
    ["mean", "std", "median"]
).reset_index()

# Ceiling effect (% score == 5): useful diagnostic for the professor's concern.
ceiling_rows = []
for method in METHODS:
    d = judgments_df[judgments_df["method"].astype(str) == method]
    for metric in METRICS:
        ceiling_rows.append({
            "method": method,
            "metric": metric,
            "n": len(d),
            "mean": d[metric].mean(),
            "std": d[metric].std(ddof=1),
            "pct_score_5": 100 * (d[metric] == 5).mean(),
            "unique_scores": int(d[metric].nunique()),
        })
diagnostics_df = pd.DataFrame(ceiling_rows)

# Same-score diagnostic: how often all 3 dimensions receive exactly same score?
judgments_df["all_three_same"] = (
    (judgments_df["clarity"] == judgments_df["evidence_integration"]) &
    (judgments_df["clarity"] == judgments_df["calibrated_usefulness"])
)

same_score_df = judgments_df.groupby("method", observed=False)["all_three_same"].agg(
    n="count", same_n="sum", same_rate="mean"
).reset_index()
same_score_df["same_rate_pct"] = same_score_df["same_rate"] * 100

# Paired wide files for later repeated-measures tests
paired = judgments_df.pivot(
    index="case_id",
    columns="method",
    values=METRICS
)

In [ ]:
# -----------------------------
# 10. Optional repeated-measures statistics

In [ ]:
# -----------------------------
# Friedman test is robust for paired ordinal 1-5 ratings across 4 methods.
stats_rows = []
try:
    from scipy.stats import friedmanchisquare

    for metric in METRICS:
        wide = judgments_df.pivot(index="case_id", columns="method", values=metric)
        wide = wide.dropna()
        if all(m in wide.columns for m in METHODS) and len(wide) >= 3:
            stat, p = friedmanchisquare(*(wide[m].values for m in METHODS))
            stats_rows.append({
                "metric": metric,
                "test": "Friedman",
                "n_cases": len(wide),
                "statistic": stat,
                "p_value": p,
            })
except Exception as ex:
    print("Friedman test skipped:", ex)

stats_df = pd.DataFrame(stats_rows)

In [ ]:
# -----------------------------
# 11. Save

In [ ]:
# -----------------------------
atomic_csv_save(explanations_df, os.path.join(RQ3_OUTPUT_DIR, "rq3_explanations.csv"))
atomic_csv_save(judgments_df, os.path.join(RQ3_OUTPUT_DIR, "rq3_judgments.csv"))
atomic_csv_save(summary, os.path.join(RQ3_OUTPUT_DIR, "rq3_summary.csv"))
atomic_csv_save(composite_summary, os.path.join(RQ3_OUTPUT_DIR, "rq3_composite_summary.csv"))
atomic_csv_save(diagnostics_df, os.path.join(RQ3_OUTPUT_DIR, "rq3_ceiling_diagnostics.csv"))
atomic_csv_save(same_score_df, os.path.join(RQ3_OUTPUT_DIR, "rq3_same_score_diagnostics.csv"))
if len(stats_df):
    atomic_csv_save(stats_df, os.path.join(RQ3_OUTPUT_DIR, "rq3_friedman_test.csv"))

print("\n" + "="*80)
print("RQ3 COMPLETE")
print("="*80)
print("\n[Primary metrics]")
print("1) Clarity               : Is the explanation clear and easy to understand?")
print("2) Evidence Integration  : Does the explanation integrate SHAP/model evidence rather than merely list it?")
print("3) Calibrated Usefulness : Does it help interpret the model decision without exaggeration?")
print("\n[Separate from RQ2]")
print("Unsupported Claim / ESR / UCR / CR / ECare evaluated in RQ2 and are not repeated in RQ3.")
print("\n[Methods]")
print("Numerical            = prediction + probability")
print("Feature Attribution  = prediction + probability + raw Top-K SHAP")
print("Template             = Use a fixed template to explain the same SHAP evidence")
print("HCLA                 = Integrate the same SHAP evidence into an HCLA justification")
print("\n[Summary]")
print(summary.round(3).to_string(index=False))
print("\n[Ceiling diagnostics]")
print(diagnostics_df.round(3).to_string(index=False))
print("\n[Same-score diagnostics]")
print(same_score_df.round(3).to_string(index=False))
if len(stats_df):
    print("\n[Friedman]")
    print(stats_df.round(5).to_string(index=False))
print("="*80)